[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imsharad/ghl-support-slm/blob/main/notebooks/v3_candidate04_colab.ipynb)

# GHL support SLM v3 candidate04: free-T4 training reproduction

This notebook reconstructs the public v3 candidate04 corpus, runs the gated QLoRA smoke, and trains 120 updates on a free Colab T4. It requires no secret and publishes nothing.

**Important:** candidate04 is a preserved failed experiment. All four checkpoints failed development selection, so it did not replace the submitted candidate03 model and must not be presented as the v3 result. The original completed run took 519.8 seconds and peaked at 3.32 GB GPU memory; a new free-tier runtime may differ or be reclaimed.

## 1. Confirm a CUDA runtime

In Colab choose **Runtime → Change runtime type → T4 GPU** before running all cells.

In [ ]:
import json, os, pathlib, shutil, subprocess, time

def run(command):
    print("RUN", command, flush=True)
    started = time.time()
    subprocess.run(command, shell=True, check=True)
    print(f"DONE in {time.time() - started:.1f}s", flush=True)

run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")

## 2. Clone the public default branch and install the locked training environment

In [ ]:
REPO = "https://github.com/Imsharad/ghl-support-slm.git"
ROOT = pathlib.Path("/content/ghl-support-slm")
if not ROOT.exists():
    run(f"git clone --depth 1 {REPO} {ROOT}")
os.chdir(ROOT)
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1", TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
print("source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
run("python -m pip install -q uv==0.8.8")
run("uv sync --frozen --extra train --python 3.11.11")
run("uv run python -c \"import torch; print(torch.__version__, torch.version.cuda); assert torch.cuda.is_available()\"")

## 3. Reconstruct the screened v3 source pool

This rebuilds the historical v2 inputs, computes the pinned MiniLM overlap screen, and removes complete near-holdout groups. It downloads the public Bitext data and pinned embedding model. Generated data stays inside this Colab VM.

In [ ]:
run("uv run python data/fetch.py")
run("uv run python data/prepare.py --placeholder-mode substitute --out data/v2 --admissions data/v2/admissions.jsonl")
run("uv run python data/audit_overlap_v3.py --source-dir data/processed/v2 --output /content/v3-source-overlap.json --allow-download")
run("uv run python data/prepare_source_v3.py --source-dir data/processed/v2 --source-audit /content/v3-source-overlap.json --output-dir /content/v3-source-pool")

## 4. Assemble candidate04 and verify its recorded hashes

Candidate04 keeps candidate03's 243 training rows and adds 18 development-driven boundary examples. The 54 validation rows are unchanged.

In [ ]:
run("uv run python data/assemble_v3.py --source-dir /content/v3-source-pool --selection data/v3/curation-queue/selection.json --targets-dir data/v3/targets --prompt-file configs/prompt-v3.txt --output-dir data/processed/v3-candidate04 --max-length 512 --augmentation data/v3/augmentation_candidate04.json --augmentation-audit data/v3/augmentation_candidate04_audit01.json --review-note 'Candidate04 data-only follow-up; development-driven assistant-authored targets; no independent human review or final-set training.'")
manifest = json.loads(pathlib.Path("data/processed/v3-candidate04/manifest.json").read_text())
expected = json.loads(pathlib.Path("data/v3/candidate04_manifest.json").read_text())
assert manifest["output_sha256"] == expected["output_sha256"]
print(json.dumps({"rows": manifest["final_rows"], "output_sha256": manifest["output_sha256"]}, indent=2))

## 5. Inspect the frozen recipe

In [ ]:
import yaml
config = yaml.safe_load(pathlib.Path("configs/training/train-v3-candidate04-t4.yaml").read_text())
print(yaml.safe_dump(config, sort_keys=False))

## 6. Run the 20-step smoke and resume proof

The smoke trains 20 updates, resumes from step 10, and requires the repeated losses to match within the fixed 0.01 tolerance. Training below should not start if this cell fails.

In [ ]:
run("uv run --extra train python train/train.py --config configs/training/train-v3-candidate04-t4.yaml --smoke --run-name v3-candidate04-colab-smoke --no-push")
smoke = json.loads(pathlib.Path("train/runs/v3-candidate04-colab-smoke/smoke.json").read_text())
print(json.dumps(smoke, indent=2))
assert smoke["ok"]

## 7. Train 120 updates

The configuration saves checkpoints at steps 30, 60, 90, and 120. Rerun this cell with `--resume` if the Python process stops but the Colab VM and run directory survive.

In [ ]:
run("uv run --extra train python train/train.py --config configs/training/train-v3-candidate04-t4.yaml --run-name v3-candidate04-colab-run --no-push")

## 8. Check, plot, and download the run

A completed training run is not a selected model. The original candidate04 checkpoints were all rejected after a separate complete development review; see `docs/v3/SELECTION.md`.

In [ ]:
RUN_DIR = pathlib.Path("train/runs/v3-candidate04-colab-run")
run(f"uv run python tools/training/check_run.py {RUN_DIR} --no-generate")
run(f"uv run python train/plot_curves.py {RUN_DIR}")
from IPython.display import Image, display
display(Image(str(RUN_DIR / "curves.png"), width=800))
archive = shutil.make_archive("/content/v3-candidate04-colab-run", "zip", RUN_DIR)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download the archive manually from the Colab file browser.")

## Recorded outcome of the original candidate04 run

The public execution record is [`data/v3/candidate04_colab_execution.json`](https://github.com/Imsharad/ghl-support-slm/blob/main/data/v3/candidate04_colab_execution.json). It records a successful free-T4 run and recovery, followed by a separate 330-response MPS development review. Checkpoints 30, 60, 90, and 120 were all ineligible because safety, boundary, or helpfulness failures remained. No final evaluation was run and the served candidate03 model was not changed.